# Day 23: RAG Pipeline Development

**Name:** Areeba Amjad  

**Date:** 27 August 2026 

**Topic:** RAG Pipeline Development


## What is RAG?

Retrieval-Augmented Generation (RAG) is a technique that combines information
retrieval with language generation. Instead of relying only on an LLM's
internal knowledge, RAG retrieves relevant information from an external
knowledge base and provides it as context to the model.

## RAG Workflow

### 1. Index

Documents → Load → Chunk → Embed → Store

### 2. Retrieve

User Query → Query Representation → Similarity Search → Top-K Documents

### 3. Generate

Retrieved Documents → Context → LLM → Final Answer

### Complete Flow

Documents
↓
Index
↓
Retrieve
↓
Context
↓
Generate
↓
Answer

# Day 23: RAG Pipeline Development

## What is RAG?

Retrieval-Augmented Generation (RAG) is a technique that combines information
retrieval with language generation. Instead of relying only on an LLM's
internal knowledge, RAG retrieves relevant information from an external
knowledge base and provides it as context to the model.

## RAG Workflow

### 1. Index

Documents → Load → Chunk → Embed → Store

### 2. Retrieve

User Query → Query Representation → Similarity Search → Top-K Documents

### 3. Generate

Retrieved Documents → Context → LLM → Final Answer

### Complete Flow

Documents
↓
Index
↓
Retrieve
↓
Context
↓
Generate
↓
Answer

## RAG vs Fine-Tuning

| Feature | RAG | Fine-Tuning |
|---|---|---|
| Main purpose | Add external knowledge | Change model behavior |
| Dynamic information | Excellent | Difficult |
| Private documents | Very suitable | Requires training data |
| Updating knowledge | Update documents/index | Retrain model |
| Style/behavior | Limited | Excellent |
| Knowledge grounding | Strong | Not guaranteed |
| Training required | Usually no model training | Yes |

### Rule of Thumb

Use **RAG** when the main problem is access to changing or external knowledge.

Use **Fine-Tuning** when the main problem is model behavior, style,
format, or task-specific behavior.

## LangChain RAG Components

### 1. VectorStore

Stores document embeddings and supports similarity search.

Examples:

- FAISS
- Chroma
- Pinecone
- Weaviate

### 2. Retriever

A retriever receives a user query and returns relevant documents.

Example:

```python
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

In [1]:

documents = [
    """
    Retrieval-Augmented Generation, or RAG, is an AI technique that combines
    information retrieval with language generation. It retrieves relevant
    information from external documents and provides that information to a
    language model as context.
    """,

    """
    A typical RAG pipeline contains three major stages: indexing, retrieval,
    and generation. During indexing, documents are loaded, split into chunks,
    converted into vector representations, and stored. During retrieval,
    relevant chunks are searched using the user query. During generation,
    the retrieved context is provided to a language model.
    """,

    """
    RAG is useful for reducing unsupported answers and grounding responses
    in external knowledge. It can work with private company documents,
    technical manuals, research papers, and other domain-specific information.
    """,

    """
    RAG and fine-tuning solve different problems. RAG is generally preferred
    when information changes frequently or must come from external documents.
    Fine-tuning is more suitable when the goal is to change model behavior,
    style, formatting, or task-specific responses.
    """,

    """
    RAG quality depends on retrieval precision, context relevance, and answer
    accuracy. If irrelevant documents are retrieved, the generated answer
    may also be incorrect. Evaluation should therefore consider both retrieval
    and generation quality.
    """
]

print("Number of documents:", len(documents))

Number of documents: 5


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english"
)

document_vectors = vectorizer.fit_transform(documents)

print("TF-IDF index created successfully!")
print("Vector shape:", document_vectors.shape)

TF-IDF index created successfully!
Vector shape: (5, 79)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_documents(query, top_k=2):
    
    query_vector = vectorizer.transform([query])
    
    similarities = cosine_similarity(
        query_vector,
        document_vectors
    )[0]
    
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    
    for index in top_indices:
        results.append({
            "document": documents[index],
            "score": float(similarities[index])
        })
    
    return results

print("Retriever created successfully!")

Retriever created successfully!


In [4]:
query = "What is RAG?"

results = retrieve_documents(query, top_k=2)

for i, result in enumerate(results, start=1):
    print(f"Result {i}")
    print("Similarity:", round(result["score"], 4))
    print(result["document"])
    print("-" * 60)

Result 1
Similarity: 0.1841

    RAG and fine-tuning solve different problems. RAG is generally preferred
    when information changes frequently or must come from external documents.
    Fine-tuning is more suitable when the goal is to change model behavior,
    style, formatting, or task-specific responses.
    
------------------------------------------------------------
Result 2
Similarity: 0.1168

    RAG is useful for reducing unsupported answers and grounding responses
    in external knowledge. It can work with private company documents,
    technical manuals, research papers, and other domain-specific information.
    
------------------------------------------------------------


In [5]:
def rag_retrieve(query, top_k=2):
    
    results = retrieve_documents(
        query,
        top_k=top_k
    )
    
    context = "\n\n".join(
        result["document"]
        for result in results
    )
    
    return context

print("RAG retrieval function ready!")

RAG retrieval function ready!


In [6]:
def rag_pipeline(query, top_k=2):
    
    # Retrieve relevant documents
    results = retrieve_documents(
        query,
        top_k=top_k
    )
    
    # Build context
    context = "\n\n".join(
        result["document"]
        for result in results
    )
    
    # Lightweight local answer
    answer = (
        "Based on the retrieved context:\n\n"
        + context.strip()
    )
    
    return answer

In [7]:
query = "What is RAG and how does it work?"

answer = rag_pipeline(
    query,
    top_k=2
)

print("Question:")
print(query)

print("\nRAG Answer:")
print(answer)

Question:
What is RAG and how does it work?

RAG Answer:
Based on the retrieved context:

RAG is useful for reducing unsupported answers and grounding responses
    in external knowledge. It can work with private company documents,
    technical manuals, research papers, and other domain-specific information.
    


    RAG and fine-tuning solve different problems. RAG is generally preferred
    when information changes frequently or must come from external documents.
    Fine-tuning is more suitable when the goal is to change model behavior,
    style, formatting, or task-specific responses.


In [8]:
questions = [
    "What is RAG?",
    "Why is RAG useful?",
    "What is the difference between RAG and fine-tuning?",
    "What are the quality challenges in RAG?"
]

for question in questions:
    
    print("=" * 70)
    print("QUESTION:", question)
    
    answer = rag_pipeline(
        question,
        top_k=2
    )
    
    print("\nANSWER:")
    print(answer)
    print()

QUESTION: What is RAG?

ANSWER:
Based on the retrieved context:

RAG and fine-tuning solve different problems. RAG is generally preferred
    when information changes frequently or must come from external documents.
    Fine-tuning is more suitable when the goal is to change model behavior,
    style, formatting, or task-specific responses.
    


    RAG is useful for reducing unsupported answers and grounding responses
    in external knowledge. It can work with private company documents,
    technical manuals, research papers, and other domain-specific information.

QUESTION: Why is RAG useful?

ANSWER:
Based on the retrieved context:

RAG is useful for reducing unsupported answers and grounding responses
    in external knowledge. It can work with private company documents,
    technical manuals, research papers, and other domain-specific information.
    


    RAG and fine-tuning solve different problems. RAG is generally preferred
    when information changes frequently or must 

In [9]:
test_query = "What is the difference between RAG and fine-tuning?"

results = retrieve_documents(
    test_query,
    top_k=3
)

print("Retrieval Quality Check")
print("=" * 50)

for i, result in enumerate(results, start=1):
    
    print(f"Document {i}")
    print("Similarity Score:",
          round(result["score"], 4))
    
    print("Relevant:",
          result["score"] > 0)
    
    print()

Retrieval Quality Check
Document 1
Similarity Score: 0.5767
Relevant: True

Document 2
Similarity Score: 0.0373
Relevant: True

Document 3
Similarity Score: 0.0352
Relevant: True



## RAG Quality Challenges

### 1. Retrieval Precision

The retriever should return documents that are actually relevant to the
user's question.

Poor retrieval can provide irrelevant context.

### 2. Context Relevance

Even if a document is retrieved, all of its content may not be useful.
Good chunking and retrieval help provide focused context.

### 3. Answer Accuracy

The generated answer should correctly reflect the retrieved evidence.

### 4. Retrieval Failure

If the knowledge base does not contain the required information, the system
may fail to answer the question correctly.

### 5. Hallucination

RAG can reduce unsupported answers but does not guarantee that hallucinations
will disappear.

### 6. Evaluation

A RAG system should be evaluated on both retrieval and generation quality.

# Complete RAG Pipeline

## Indexing

Documents
↓
Text Preparation
↓
TF-IDF Vectorization
↓
Vector Representation

## Retrieval

User Query
↓
Query Vector
↓
Cosine Similarity
↓
Top-K Relevant Documents

## Generation

Retrieved Documents
↓
Context
↓
Answer Generation
↓
Final Answer

## Complete Pipeline

Documents → Index → Retrieve → Context → Generate → Answer

## Implementation

This notebook implements a lightweight RAG demonstration using:

- Python
- TF-IDF Vectorization
- Cosine Similarity
- Top-K Retrieval
- Context Construction

A production RAG system can replace TF-IDF with transformer-based embeddings
and use vector stores such as FAISS, Chroma, Pinecone, or Weaviate.

## Conclusion

RAG allows an AI system to retrieve relevant external information before
generating an answer. This makes it particularly useful for dynamic,
private, and domain-specific knowledge.

RAG is generally preferred when knowledge changes frequently, while
fine-tuning is more appropriate when changing model behavior, style, or
task-specific behavior is the primary goal.

# Research Sources

## ChatGPT
Used for understanding RAG architecture, RAG vs fine-tuning, LangChain
components, and common RAG quality challenges.

## Gemini
Used as a secondary source for comparing RAG workflows, retrieval methods,
and practical RAG applications.

## Claude
Used as a secondary source for understanding RAG limitations, hallucination
reduction, and evaluation considerations.

## RAG Articles / Research

1. Lewis et al. — Retrieval-Augmented Generation for Knowledge-Intensive
   NLP Tasks (2020)

2. Gao et al. — Retrieval-Augmented Generation for Large Language Models:
   A Survey (2023)

3. Wu et al. — Retrieval-augmented generation for natural language
   processing: a survey (2026)

## Documentation

- LangChain Retrieval Documentation
- LangChain Semantic Search / Knowledge Base Documentation

# Task 1: Simple RAG Pipeline

The goal is to build a basic Retrieval-Augmented Generation system.

Pipeline:

Documents
↓
Chunking
↓
Embeddings
↓
FAISS Vector Store
↓
Retriever
↓
Relevant Documents
↓
Context
↓
Answer

In [1]:
import importlib.util

packages = {
    "faiss": "FAISS",
    "sentence_transformers": "Sentence Transformers",
    "langchain_text_splitters": "LangChain Text Splitters"
}

for package, name in packages.items():
    print(
        f"{name}:",
        "Installed" if importlib.util.find_spec(package) else "Missing"
    )

FAISS: Installed
Sentence Transformers: Installed
LangChain Text Splitters: Installed


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

In [3]:
documents = [
    {
        "text": """
        Retrieval-Augmented Generation, commonly called RAG, combines
        information retrieval with language generation. RAG retrieves
        relevant information from external documents and provides that
        information to a language model as context before generating an answer.
        """,
        "source": "rag_intro.txt"
    },

    {
        "text": """
        A typical RAG system has three major stages: indexing, retrieval,
        and generation. During indexing, documents are loaded, split into
        smaller chunks, converted into embeddings, and stored in a vector
        database. During retrieval, a user query is compared with stored
        vectors and the most relevant chunks are selected.
        """,
        "source": "rag_workflow.txt"
    },

    {
        "text": """
        RAG can reduce unsupported answers by grounding generation in
        retrieved documents. It is useful for private company information,
        technical documentation, research papers, manuals, and other
        domain-specific knowledge.
        """,
        "source": "rag_benefits.txt"
    },

    {
        "text": """
        RAG and fine-tuning solve different problems. RAG is useful when
        external knowledge changes frequently. Fine-tuning is more suitable
        when the goal is to change model behavior, style, formatting, or
        task-specific behavior.
        """,
        "source": "rag_vs_finetuning.txt"
    },

    {
        "text": """
        RAG quality depends on retrieval precision, context relevance,
        and answer accuracy. If irrelevant chunks are retrieved, the
        generated answer may also be incorrect. Therefore retrieval quality
        should be evaluated separately from generation quality.
        """,
        "source": "rag_quality.txt"
    }
]

print("Documents loaded:", len(documents))

Documents loaded: 5


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = []

for doc in documents:
    split_texts = splitter.split_text(doc["text"])
    
    for i, chunk in enumerate(split_texts):
        chunks.append({
            "text": chunk,
            "source": doc["source"],
            "chunk_id": i
        })

print("Total chunks:", len(chunks))

Total chunks: 6


In [5]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
)

embeddings = np.array(embeddings).astype("float32")

print("Embedding shape:", embeddings.shape)

Embedding shape: (6, 384)


In [6]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS vector store created successfully!")
print("Vectors stored:", index.ntotal)

FAISS vector store created successfully!
Vectors stored: 6


In [7]:
search_kwargs = {"k": 4}

In [8]:
search_kwargs = {
    "k": 4
}

def retrieve(query, search_kwargs=search_kwargs):
    
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )
    
    query_embedding = np.array(
        query_embedding
    ).astype("float32")
    
    scores, indices = index.search(
        query_embedding,
        search_kwargs["k"]
    )
    
    results = []
    
    for score, idx in zip(scores[0], indices[0]):
        
        if idx != -1:
            result = chunks[idx].copy()
            result["score"] = float(score)
            results.append(result)
    
    return results

print("Retriever created successfully!")

Retriever created successfully!


In [9]:
query = "What is RAG?"

results = retrieve(query)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Chunk:", result["chunk_id"])
    print("Score:", round(result["score"], 4))
    print("Text:", result["text"])


Result 1
Source: rag_benefits.txt
Chunk: 0
Score: 0.6279
Text: RAG can reduce unsupported answers by grounding generation in
        retrieved documents. It is useful for private company information,
        technical documentation, research papers, manuals, and other
        domain-specific knowledge.

Result 2
Source: rag_workflow.txt
Chunk: 0
Score: 0.5524
Text: A typical RAG system has three major stages: indexing, retrieval,
        and generation. During indexing, documents are loaded, split into
        smaller chunks, converted into embeddings, and stored in a vector
        database. During retrieval, a user query is compared with stored

Result 3
Source: rag_vs_finetuning.txt
Chunk: 0
Score: 0.5505
Text: RAG and fine-tuning solve different problems. RAG is useful when
        external knowledge changes frequently. Fine-tuning is more suitable
        when the goal is to change model behavior, style, formatting, or
        task-specific behavior.

Result 4
Source: rag_intro.t

In [10]:
def simple_rag_answer(query, top_k=4):
    
    results = retrieve(
        query,
        {"k": top_k}
    )
    
    context = format_context(results)
    
    answer = (
        "Answer based on retrieved documents:\n\n"
        + context
    )
    
    return answer, results

In [ ]:
query = "Why is RAG useful

answer, sources = simple_rag_answer(query)

print(answer)

## Task 2 — Enhanced RAG

In [5]:
def format_context(results):
    context_parts = []

    for result in results:
        context_parts.append(
            f"Source: {result['source']}\n"
            f"{result['document']}"
        )

    return "\n\n".join(context_parts)


print("Context formatter ready!")

Context formatter ready!


In [8]:
PROMPT_TEMPLATE = """
Answer based ONLY on the context below.

If the answer is not available in the context,
say: "I don't have that information."

Context:
{context}

Question:
{question}

Answer:
"""

print(PROMPT_TEMPLATE


Answer based ONLY on the context below.

If the answer is not available in the context,
say: "I don't have that information."

Context:
{context}

Question:
{question}

Answer:



In [9]:
def enhanced_rag(query, top_k=4):

    # Retrieve relevant documents
    results = retrieve(query, top_k)

    # Format retrieved context
    context = format_context(results)

    # Create prompt
    prompt = PROMPT_TEMPLATE.format(
        context=context,
        question=query
    )

    return {
        "query": query,
        "prompt": prompt,
        "results": results
    }


print("Enhanced RAG ready!")

Enhanced RAG ready!


## Task 3 — Source Attribution

In [2]:
# ============================================================
# DAY 23 - TASK 3: MULTI-DOCUMENT RAG SYSTEM
# Complete Task 3 in ONE CELL
# ============================================================

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------
# 1. MULTIPLE DOCUMENT SOURCES
# ------------------------------------------------------------

documents = [
    {
        "text": """
        Retrieval-Augmented Generation (RAG) combines document retrieval
        with language generation. Documents are divided into chunks,
        converted into vector representations, and stored for retrieval.
        """,
        "source": "rag_overview.txt",
        "source_type": "text",
        "domain": "Artificial Intelligence",
        "date": "2026-08-27"
    },

    {
        "text": """
        RAG can reduce hallucinations by providing relevant external
        information to a language model. It is useful for private company
        documents, internal knowledge bases, and frequently changing data.
        """,
        "source": "rag_benefits.pdf",
        "source_type": "pdf",
        "domain": "Artificial Intelligence",
        "date": "2026-08-26"
    },

    {
        "text": """
        RAG and fine-tuning solve different problems. RAG is useful for
        dynamic and external knowledge. Fine-tuning is mainly useful for
        changing model behavior, style, formatting, or task performance.
        """,
        "source": "rag_vs_finetuning.txt",
        "source_type": "text",
        "domain": "Machine Learning",
        "date": "2026-08-25"
    },

    {
        "text": """
        A RAG workflow contains indexing, retrieval, and generation.
        During indexing documents are prepared and embedded. During
        retrieval relevant chunks are searched. During generation the
        retrieved context is used to produce an answer.
        """,
        "source": "rag_workflow.pdf",
        "source_type": "pdf",
        "domain": "Artificial Intelligence",
        "date": "2026-08-24"
    },

    {
        "text": """
        RAG quality depends on retrieval precision, context relevance,
        and answer accuracy. Poor retrieval can result in irrelevant
        context and incorrect answers.
        """,
        "source": "rag_quality.txt",
        "source_type": "text",
        "domain": "Artificial Intelligence",
        "date": "2026-08-23"
    },

    {
        "text": """
        LangChain provides vector stores, retrievers, document loaders,
        prompt templates, and retrieval chains for building RAG systems.
        A retriever searches a vector store and returns relevant documents.
        """,
        "source": "langchain_rag.html",
        "source_type": "website",
        "domain": "LangChain",
        "date": "2026-08-22"
    }
]


# ------------------------------------------------------------
# 2. CHUNK DOCUMENTS + PRESERVE METADATA
# ------------------------------------------------------------

chunks = []

for doc in documents:

    words = doc["text"].split()

    # Small chunks for demonstration
    for i in range(0, len(words), 60):

        chunk = " ".join(words[i:i + 60])

        chunks.append({
            "document": chunk,
            "source": doc["source"],
            "source_type": doc["source_type"],
            "domain": doc["domain"],
            "date": doc["date"]
        })


print("Documents:", len(documents))
print("Chunks created:", len(chunks))


# ------------------------------------------------------------
# 3. VECTOR REPRESENTATION
# ------------------------------------------------------------

texts = [chunk["document"] for chunk in chunks]

vectorizer = TfidfVectorizer(
    stop_words="english"
)

document_vectors = vectorizer.fit_transform(texts)

print("Vector index created successfully!")


# ------------------------------------------------------------
# 4. FILTERED RETRIEVAL FUNCTION
# ------------------------------------------------------------

def task3_retrieve(query, top_k=4, source_type=None):

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        document_vectors
    )[0]

    candidates = []

    for i, chunk in enumerate(chunks):

        # Apply source-type filter
        if source_type is None:
            candidates.append((i, scores[i]))

        elif chunk["source_type"] == source_type:
            candidates.append((i, scores[i]))

    # Highest relevance first
    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for index, score in candidates[:top_k]:

        results.append({
            "document": chunks[index]["document"],
            "source": chunks[index]["source"],
            "source_type": chunks[index]["source_type"],
            "domain": chunks[index]["domain"],
            "date": chunks[index]["date"],
            "score": float(score)
        })

    return results


# ------------------------------------------------------------
# 5. MULTI-DOCUMENT ANSWER SYNTHESIS
# ------------------------------------------------------------

def task3_rag(
    query,
    top_k=4,
    source_type=None
):

    results = task3_retrieve(
        query=query,
        top_k=top_k,
        source_type=source_type
    )

    print("\n" + "=" * 70)
    print("MULTI-DOCUMENT RAG SYSTEM")
    print("=" * 70)

    print("\nQuestion:")
    print(query)

    if source_type:
        print("\nSource Type Filter:")
        print(source_type)

    # No relevant documents
    relevant = [
        item for item in results
        if item["score"] > 0
    ]

    if not relevant:

        print("\nAnswer:")
        print("I don't have that information.")

        print("\nSources:")
        print("No relevant sources found.")

        return

    # --------------------------------------------------------
    # Answer synthesis
    # --------------------------------------------------------

    print("\nAnswer:")
    
    print(
        "Based on the retrieved documents, "
        "the relevant information is:"
    )

    for item in relevant:

        print(
            "\n- " + item["document"]
        )

    # --------------------------------------------------------
    # Sources + relevance scores
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("Sources and Relevance Scores")
    print("-" * 70)

    for i, item in enumerate(relevant, 1):

        print(
            f"{i}. {item['source']}"
        )

        print(
            f"   Source Type: {item['source_type']}"
        )

        print(
            f"   Domain: {item['domain']}"
        )

        print(
            f"   Date: {item['date']}"
        )

        print(
            f"   Relevance Score: {item['score']:.4f}"
        )


# ------------------------------------------------------------
# 6. TEST 1 — ALL DOCUMENT SOURCES
# ------------------------------------------------------------

task3_rag(
    "Why is RAG useful for private data?",
    top_k=4
)


# ------------------------------------------------------------
# 7. TEST 2 — PDF ONLY
# ------------------------------------------------------------

task3_rag(
    "What is the RAG workflow?",
    top_k=4,
    source_type="pdf"
)


# ------------------------------------------------------------
# 8. TEST 3 — TEXT ONLY
# ------------------------------------------------------------

task3_rag(
    "What is the difference between RAG and fine-tuning?",
    top_k=4,
    source_type="text"
)


# ------------------------------------------------------------
# 9. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TASK 3 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("""
✓ Multiple document sources indexed
✓ PDF sources included
✓ Website source included
✓ Text files included
✓ Metadata added
✓ source_type metadata added
✓ domain metadata added
✓ date metadata added
✓ Vector retrieval implemented
✓ Filtered retrieval implemented
✓ Multi-document synthesis implemented
✓ Relevance scores displayed
✓ Source attribution implemented
""")

Documents: 6
Chunks created: 6
Vector index created successfully!

MULTI-DOCUMENT RAG SYSTEM

Question:
Why is RAG useful for private data?

Answer:
Based on the retrieved documents, the relevant information is:

- RAG can reduce hallucinations by providing relevant external information to a language model. It is useful for private company documents, internal knowledge bases, and frequently changing data.

- RAG and fine-tuning solve different problems. RAG is useful for dynamic and external knowledge. Fine-tuning is mainly useful for changing model behavior, style, formatting, or task performance.

- RAG quality depends on retrieval precision, context relevance, and answer accuracy. Poor retrieval can result in irrelevant context and incorrect answers.

- Retrieval-Augmented Generation (RAG) combines document retrieval with language generation. Documents are divided into chunks, converted into vector representations, and stored for retrieval.

-----------------------------------------